# Session 25 — Semantic Layering & Metadata Preparation

**Dataset:** Online Retail (UK online gift retailer, Dec-2010 → Dec-2011), 1 % sample, ~5.4 k invoice lines.

### What you will learn
| # | Concept | What you will *see* in this notebook |
|---|---------|--------------------------------------|
| 1 | **Metadata** | Technical (schema, location, format), business (comments, owners, definitions) and operational (statistics, data-quality counts) metadata attached to tables |
| 2 | **Metastore / Catalog** | Where Spark keeps that metadata, how to query it, and proof that it survives a restart |
| 3 | **Managed vs. unmanaged (external) tables** | Who owns the files, what `DROP` / `TRUNCATE` / `RENAME` really do, and why partitions can "disappear" |
| 4 | **Semantic layering** | Bronze → Silver → Gold layers, and a Gold *semantic layer* of business-friendly dimensions, facts and governed metrics |

### Agenda
0. Big picture & setup
1. Load the data, temp views (session-only metadata)
2. The metastore & catalog
3. Managed vs. external tables — four experiments
4. Bronze layer — external table over raw files (schema-on-read)
5. Silver layer — cleaned, typed, documented managed table
6. Metadata deep dive — statistics and the optimizer
7. Gold layer — the semantic layer
8. Metadata as data — generating a data dictionary
9. Wrap-up & proof of persistence
10. Exercises

> **Requirements:** Spark 3.5+ (pip `pyspark` already ships with Hive support). The CSV is expected at `data/online-retail-dataset-one-percent-sample.csv` relative to the notebook.

## Part 0 — The big picture

When you run `SELECT * FROM silver.sales_lines`, Spark has to answer two very different questions:

1. **"What is `silver.sales_lines`?"** — which columns, types, file format, where the files are, how it is partitioned, how big it is. That is **metadata**, and it lives in the **metastore**.
2. **"Give me the bytes."** — that is **data**, and it lives in **storage** (local disk here; S3 / ADLS / GCS / HDFS in production).

```
                 SQL / DataFrame API
                         │
              ┌──────────▼───────────┐
              │  Spark driver        │
              │  parser → analyzer → │   asks the CATALOG: "what & where is it?"
              │  optimizer → planner │──────────────────────┐
              └──────────┬───────────┘                      │
                         │ reads/writes bytes               ▼
    ┌────────────────────▼──────────────────┐   ┌──────────────────────────────┐
    │  STORAGE  (files)                     │   │  METASTORE  (metadata)       │
    │  spark-warehouse/   ← MANAGED tables  │   │  databases, tables, columns, │
    │  external-data/     ← EXTERNAL tables │   │  types, format, location,    │
    │  landing/           ← raw source files│   │  partitions, statistics,     │
    │                                       │   │  comments, properties,       │
    │                                       │   │  view SQL text               │
    └───────────────────────────────────────┘   │  here: ./metastore_db (Derby)│
                                                └──────────────────────────────┘
```

**Key vocabulary**

* **Metadata** — data about data. We will use three flavours:
  * *Technical*: schema, data types, file format, location, partitions.
  * *Business*: descriptions, owners, stewards, definitions, sensitivity tags.
  * *Operational*: row counts, sizes, column statistics, load timestamps, data-quality results.
* **Metastore** — the database that *stores* metadata. Locally Spark uses an embedded Apache Derby database (`./metastore_db`). In production it is a Hive Metastore backed by MySQL/Postgres, AWS Glue Data Catalog, Databricks Unity Catalog, or an Iceberg REST catalog (e.g. Polaris).
* **Catalog** — the API/namespace through which Spark *reads* the metastore (`spark.catalog`, `SHOW TABLES`, `DESCRIBE …`). Objects are addressed as `catalog.database.table`.
* **Managed table** — Spark owns **both** metadata **and** data. Files live in the warehouse directory. `DROP TABLE` deletes the files.
* **External (unmanaged) table** — Spark owns **only** the metadata; it points at files at a `LOCATION` you control. `DROP TABLE` deletes only the metadata.
* **Semantic layer** — a business-facing layer (views, dimensions, facts, metric definitions) that translates physical tables into the language of the business, so everybody computes "revenue" the same way.

### 0.1 Start Spark *with a persistent metastore*

Two changes from a plain session:

* `.enableHiveSupport()` switches the catalog from `in-memory` (metadata vanishes when the session ends) to `hive` (metadata is persisted in `./metastore_db`).
* `spark.sql.warehouse.dir` pins where **managed** tables are stored.

> ⚠️ These are *static* settings — they only apply when the session is first created. If a Spark session is already running in this kernel, restart the kernel first.

In [ ]:
import pyspark, os, sys, shutil, pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR   = Path(os.getcwd())
WAREHOUSE_DIR = PROJECT_DIR / "spark-warehouse"   # Spark decides layout here  -> MANAGED tables
EXTERNAL_DIR  = PROJECT_DIR / "external-data"     # storage that WE control     -> EXTERNAL tables
LANDING_DIR   = PROJECT_DIR / "landing"           # raw files as delivered by the source system

spark = (SparkSession.builder
         .appName("semantic-layering-and-meta-data-preparation")
         .config("spark.sql.warehouse.dir", WAREHOUSE_DIR.as_posix())
         .enableHiveSupport()          # <- persistent metastore (embedded Derby in ./metastore_db)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version          :", spark.version)
print("Catalog implementation :", spark.conf.get("spark.sql.catalogImplementation"))   # expect: hive
print("Warehouse directory    :", spark.conf.get("spark.sql.warehouse.dir"))
spark

### 0.2 Helper functions
Small utilities we will reuse to look at **metadata** (`table_summary`) and at **the files on disk** (`tree`) side by side.

In [ ]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

def sql(query, n=100):
    """Run Spark SQL and return the first n rows as a pandas DataFrame (pretty notebook display)."""
    return spark.sql(query).limit(n).toPandas()

def catalog_tables(db=None):
    """List tables/views registered in the catalog as a pandas DataFrame."""
    # note: listTables(db) also returns *local temp views*; we hide them when a db is given
    tables = spark.catalog.listTables(db) if db else spark.catalog.listTables()
    if db:
        tables = [t for t in tables if not t.isTemporary]
    return pd.DataFrame([t._asdict() for t in tables])

def table_summary(fqtn, keys=("Type", "Provider", "Location", "Comment", "Partition Provider",
                              "Table Properties", "Statistics", "View Text")):
    """Pick the interesting rows out of DESCRIBE TABLE EXTENDED."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    info = {}
    for r in rows:
        name = (r.col_name or "").strip()
        if name in keys and name not in info:
            info[name] = r.data_type
    return pd.DataFrame({"property": list(info), "value": list(info.values())}).set_index("property")

def table_location(fqtn):
    """Physical location of a table, read from the metastore."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    loc = next((r.data_type for r in rows if r.col_name == "Location"), None)
    return Path(urlparse(loc).path) if loc else None

def tree(path, max_files=3):
    """Print a directory tree (hides _SUCCESS and .crc housekeeping files). max_files=0 -> folders only."""
    path = Path(path)
    if not path.exists():
        print(f"{path}   <-- DOES NOT EXIST")
        return
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = len(Path(root).relative_to(path).parts)
        indent = "    " * depth
        print(f"{indent}{Path(root).name}/")
        files = sorted(f for f in files if not f.startswith((".", "_")))
        for f in files[:max_files]:
            print(f"{indent}    {f}  ({(Path(root) / f).stat().st_size:,} bytes)")
        if max_files and len(files) > max_files:
            print(f"{indent}    ... {len(files) - max_files} more file(s)")

def dir_size(path):
    path = Path(path) if path else None
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path and path.exists() else 0

### 0.3 Clean slate (makes the notebook re-runnable)
Drops the demo databases and deletes the demo folders. **Skip this cell** in Part 9 when we prove that metadata persists across restarts.

In [ ]:
DEMO_DATABASES = ["lab", "bronze", "silver", "gold", "governance"]
RESET = True

if RESET:
    for db in DEMO_DATABASES:
        spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
        shutil.rmtree(WAREHOUSE_DIR / f"{db}.db", ignore_errors=True)
    shutil.rmtree(EXTERNAL_DIR, ignore_errors=True)
    shutil.rmtree(LANDING_DIR, ignore_errors=True)

print("Databases now in the metastore:", [d.name for d in spark.catalog.listDatabases()])

## Part 1 — Load the data

`inferSchema` makes Spark scan the file and *guess* the types. The guessed schema is the first piece of **technical metadata** we meet. Notice the guesses: `CustomerID` becomes a `double` (because of values like `16906.0`) and `InvoiceDate` stays a `string`. We will fix both deliberately in the Silver layer.

In [ ]:
file_name = os.path.join(os.getcwd(), "data/online-retail-dataset-one-percent-sample.csv")
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_name)
df

In [ ]:
df.printSchema()

In [ ]:
df.show(3)

In [ ]:
print("rows:", df.count())
# Schema is metadata you can inspect programmatically, too
[(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields]

### 1.1 Temp views: metadata that lives only in your session

A **temporary view** is just a *name* mapped to a *query plan*, held in the memory of this `SparkSession`.
* No files are written.
* Nothing goes to the metastore → it disappears when the session ends.
* Other sessions cannot see it (a *global* temp view is visible to other sessions of the same application, under the special `global_temp` database).

In [ ]:
df.createOrReplaceTempView("sales")

# tableType = TEMPORARY, isTemporary = True, namespace is empty: it does not belong to any database
catalog_tables()

In [ ]:
sql("""
SELECT Country,
       COUNT(*)                          AS line_count,
       ROUND(SUM(Quantity * UnitPrice), 2) AS naive_revenue
FROM   sales
GROUP  BY Country
ORDER  BY naive_revenue DESC
""", 8)

In [ ]:
df.createOrReplaceGlobalTempView("sales_global")

other_session = spark.newSession()          # same application, different session
try:
    other_session.sql("SELECT COUNT(*) FROM sales").show()
except Exception as e:
    print("Local temp view from another session ->", type(e).__name__, ":", str(e).splitlines()[0][:120])

print("Global temp view from another session ->", other_session.sql("SELECT COUNT(*) AS n FROM global_temp.sales_global").first().n, "rows")

## Part 2 — The metastore & catalog

The catalog is a hierarchy: **catalog → database (a.k.a. schema / namespace) → table / view → column**.
Spark's built-in catalog is called `spark_catalog`; `default` is the database you start in.

In [ ]:
print("Catalog implementation:", spark.conf.get("spark.sql.catalogImplementation"))
print("Current catalog       :", spark.catalog.currentCatalog())
print("Current database      :", spark.catalog.currentDatabase())
pd.DataFrame([d._asdict() for d in spark.catalog.listDatabases()])

### 2.1 One database per layer
Databases are the first place to attach metadata: a **comment** and free-form **properties**. We create one database per layer of our architecture (plus a `lab` sandbox for experiments).

> Note: `owner`, `location` and `comment` are *reserved* property names in Spark, so we use `data_owner` instead.

In [ ]:
layers = {
    "lab":    ("Sandbox for the managed-vs-external experiments",                                  "sandbox"),
    "bronze": ("Raw data exactly as delivered by source systems. Append-only, schema-on-read",     "bronze"),
    "silver": ("Cleaned, typed, de-duplicated and conformed data. One row per invoice line",       "silver"),
    "gold":   ("Semantic layer: business-friendly dimensions, facts and governed metrics",         "gold"),
}

for db, (comment, layer) in layers.items():
    spark.sql(f"""
        CREATE DATABASE IF NOT EXISTS {db}
        COMMENT '{comment}'
        WITH DBPROPERTIES ('layer' = '{layer}',
                           'data_owner' = 'data-engineering-class',
                           'created_by' = 'session-25-notebook')
    """)

pd.DataFrame([d._asdict() for d in spark.catalog.listDatabases()])

In [ ]:
# Every database gets a default location inside the warehouse: <warehouse>/<db>.db
sql("DESCRIBE DATABASE EXTENDED silver")

### 2.2 Where do metadata and data physically live?
* `spark-warehouse/` holds the **data** of managed tables (one folder per database, empty for now).
* `metastore_db/` holds the **metadata** — it is a Derby database, not something you edit by hand. Its internal tables (`DBS`, `TBLS`, `SDS`, `COLUMNS_V2`, `PARTITIONS`, `TABLE_PARAMS`, …) are the same in every Hive Metastore, whether on your laptop or on a production MySQL server.
* Embedded Derby allows **one** Spark process at a time — fine for class, the reason production uses a shared metastore service.

In [ ]:
tree(WAREHOUSE_DIR)

metastore_dir = PROJECT_DIR / "metastore_db"
print("\nMetastore folder:", metastore_dir, "| exists:", metastore_dir.exists())
if metastore_dir.exists():
    print("contents:", sorted(p.name for p in metastore_dir.iterdir()))

## Part 3 — Managed vs. external (unmanaged) tables

We will save the **same DataFrame** twice, once as each type, and then run four experiments.

| | Managed | External / unmanaged |
|---|---|---|
| How you create it | `saveAsTable("db.t")` or `CREATE TABLE … USING parquet` with **no** location | `saveAsTable` with `.option("path", …)`, or `CREATE TABLE … LOCATION '…'` |
| Who chooses the location | Spark (`<warehouse>/<db>.db/<table>`) | You |
| Who owns the files | Spark | You / another system |

In [ ]:
lab_df = df.withColumn("invoice_month",
                       F.date_format(F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"), "yyyy-MM"))

# MANAGED: we only give Spark a NAME. Spark decides where the files go.
(lab_df.write
       .mode("overwrite")
       .format("parquet")
       .partitionBy("invoice_month")
       .saveAsTable("lab.sales_managed"))

# EXTERNAL: same data, but WE decide where the files go.
external_path = EXTERNAL_DIR / "lab" / "sales_external"
(lab_df.write
       .mode("overwrite")
       .format("parquet")
       .partitionBy("invoice_month")
       .option("path", external_path.as_posix())     # <- this one option makes it EXTERNAL
       .saveAsTable("lab.sales_external"))

catalog_tables("lab")

In [ ]:
# Same data, same schema — the metastore records a different Type and Location
pd.concat({t: table_summary(f"lab.{t}", keys=("Type", "Provider", "Location", "Partition Provider"))["value"]
           for t in ["sales_managed", "sales_external"]}, axis=1)

In [ ]:
managed_path  = table_location("lab.sales_managed")
external_path = table_location("lab.sales_external")

print("MANAGED table files (inside the warehouse):")
tree(managed_path)
print("\nEXTERNAL table files (outside the warehouse, in a folder we chose):")
tree(external_path)

### Experiment 1 — `DROP TABLE`
Prediction time: after dropping both tables, which folders still exist?

In [ ]:
print("BEFORE DROP")
print(f"  managed : {dir_size(managed_path):>10,} bytes  at {managed_path}")
print(f"  external: {dir_size(external_path):>10,} bytes  at {external_path}")

spark.sql("DROP TABLE lab.sales_managed")
spark.sql("DROP TABLE lab.sales_external")

print("\nAFTER DROP")
print(f"  managed  folder exists? {managed_path.exists()}   <- metadata AND data are gone")
print(f"  external folder exists? {external_path.exists()}   <- only the metadata is gone; files are 'orphaned'")
print("\nTables left in lab:", [t.name for t in spark.catalog.listTables("lab") if not t.isTemporary])

### Experiment 2 — Re-register the orphaned external files

Files without metadata are just files: Spark SQL can't find them by name. Because we still have the files, we can **re-create the metadata** and point it at them. Nothing is copied — this is a metadata-only operation.

Watch what happens with the partitions: for a partitioned table, the **list of partitions is itself metadata in the metastore** (`Partition Provider: Catalog`). A freshly created table knows the partition *column*, but not yet which partition *values* exist.

In [ ]:
# Re-declare the schema (technical metadata) — partition column goes last
columns_ddl = ",\n    ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in lab_df.schema.fields)

create_stmt = f"""
CREATE TABLE lab.sales_external (
    {columns_ddl}
)
USING PARQUET
PARTITIONED BY (invoice_month)
LOCATION '{external_path.as_posix()}'
"""
print(create_stmt)
spark.sql(create_stmt)

print("Rows visible right after CREATE TABLE :", spark.table("lab.sales_external").count())
print("Partitions known to the metastore     :", spark.sql("SHOW PARTITIONS lab.sales_external").count())

In [ ]:
# Ask Spark to scan the folder and register every partition it finds in the metastore
spark.sql("MSCK REPAIR TABLE lab.sales_external")       # equivalent: ALTER TABLE ... RECOVER PARTITIONS

print("Rows visible after MSCK REPAIR :", spark.table("lab.sales_external").count())
sql("SHOW PARTITIONS lab.sales_external")

> **Takeaway:** the data never moved. The rows were "missing" purely because the *metadata* (the partition list) was missing. Many real-world "my new data isn't showing up" tickets are exactly this.

### Experiment 3 — `TRUNCATE` and `RENAME`

In [ ]:
# Spark refuses to delete data it does not own
try:
    spark.sql("TRUNCATE TABLE lab.sales_external")
except Exception as e:
    print("TRUNCATE on EXTERNAL table refused ->", str(e).splitlines()[0][:200])

In [ ]:
# Re-create a managed copy, then rename both tables
lab_df.write.mode("overwrite").format("parquet").saveAsTable("lab.sales_managed")

before = {t: table_location(f"lab.{t}") for t in ["sales_managed", "sales_external"]}
spark.sql("ALTER TABLE lab.sales_managed  RENAME TO lab.sales_managed_v2")
spark.sql("ALTER TABLE lab.sales_external RENAME TO lab.sales_external_v2")
after = {"sales_managed": table_location("lab.sales_managed_v2"),
         "sales_external": table_location("lab.sales_external_v2")}

for t in before:
    print(f"{t:15s}\n   before: {before[t]}\n   after : {after[t]}\n")
print("-> Managed: Spark MOVED the folder to match the new name.")
print("-> External: only the name in the metastore changed; the files stayed where we put them.")

### Experiment 4 (think about it) — two tables, one location
What would happen if you created an *external* table that points at the *managed* table's folder, and then dropped the managed table? (Answer: the managed `DROP` deletes the files and the external table now points at nothing.) Rule of thumb: **one location ↔ one owner.**

### Summary

| Operation | Managed table | External table |
|---|---|---|
| `DROP TABLE` | deletes metadata **and** files | deletes metadata only — files remain |
| `TRUNCATE TABLE` | deletes the files' contents | refused |
| `ALTER TABLE … RENAME` | renames and **moves** the folder | renames the metadata only |
| `INSERT` / `INSERT OVERWRITE` | ✔ | ✔ (you are still writing into your own folder) |
| Typical use | data the platform produces and owns: Silver, Gold aggregates | data owned by someone else or shared with other engines: raw landing files, data lake folders read by other tools |
| Main risk | accidental `DROP` = data loss | orphaned files, stale partition metadata |

> In Databricks / Unity Catalog the same split exists (managed vs. external tables, with extra governance on top), and Delta/Iceberg tables add a transaction log, but the ownership idea is identical.

## Part 4 — Semantic layering: the Medallion architecture

```
   source export (CSV)            BRONZE                    SILVER                         GOLD  = semantic layer
  ┌──────────────────┐     ┌──────────────────┐     ┌──────────────────────┐     ┌──────────────────────────────┐
  │ landing/         │ ──► │ external table   │ ──► │ managed table        │ ──► │ dim_date, dim_product,       │
  │ online_retail/   │     │ over raw files   │     │ typed, cleaned,      │     │ dim_customer, fact_sales,    │
  │ *.csv            │     │ schema-on-read   │     │ de-duplicated,       │     │ sales_metrics_monthly,       │
  └──────────────────┘     │ source names     │     │ snake_case names,    │     │ metric_definitions           │
                           │ "as delivered"   │     │ flags + audit cols   │     │ business names & definitions │
                           └──────────────────┘     └──────────────────────┘     └──────────────────────────────┘
      who uses it:            data engineers           data engineers,               analysts, BI tools,
                              (replay, audit)          data scientists               business users
```

Each layer has a **contract** — and the contract is written down as metadata (comments, properties, grain, lineage).

## Part 4.1 — Bronze: an external table over the raw files

The source system "drops" CSV files in `landing/online_retail/`. Bronze does **not** copy or change them; it simply registers metadata on top of them:
* an **explicit schema** (no more guessing with `inferSchema`),
* **column comments** documenting what the source sends,
* **table properties**: owner, source system, refresh frequency, sensitivity.

This is **schema-on-read**: the schema is applied every time the files are read.

In [ ]:
bronze_landing = LANDING_DIR / "online_retail"
bronze_landing.mkdir(parents=True, exist_ok=True)
shutil.copy(file_name, bronze_landing / "online_retail_2011_part_0001.csv")   # simulate the source dropping a file
tree(LANDING_DIR)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze.online_retail_raw (
    InvoiceNo   STRING  COMMENT 'Invoice number as delivered. A leading C marks a cancellation',
    StockCode   STRING  COMMENT 'Product code. Non-numeric codes (POST, DOT, M, AMAZONFEE...) are not products',
    Description STRING  COMMENT 'Free-text product description, may be NULL',
    Quantity    INT     COMMENT 'Units on the line. Negative for cancellations and returns',
    InvoiceDate STRING  COMMENT 'Invoice date-time as text, format M/d/yyyy H:mm, UK local time',
    UnitPrice   DOUBLE  COMMENT 'Unit price in GBP',
    CustomerID  DOUBLE  COMMENT 'Customer identifier, exported as a float. NULL for guest checkouts',
    Country     STRING  COMMENT 'Ship-to country'
)
USING CSV
OPTIONS (header 'true', mode 'PERMISSIVE')
LOCATION '{bronze_landing.as_posix()}'
COMMENT 'Raw online-retail invoice lines exactly as exported by the e-commerce platform'
TBLPROPERTIES (
    'layer'             = 'bronze',
    'source_system'     = 'ecommerce_platform_export',
    'source_format'     = 'csv',
    'load_pattern'      = 'file drop into landing/online_retail',
    'data_owner'        = 'sales-operations',
    'data_steward'      = 'data-engineering-class',
    'contains_pii'      = 'false',
    'refresh_frequency' = 'daily'
)
""")

table_summary("bronze.online_retail_raw")

In [ ]:
# Column-level metadata: names, types, comments
sql("DESCRIBE TABLE bronze.online_retail_raw")

In [ ]:
sql("SELECT * FROM bronze.online_retail_raw LIMIT 5")

### 4.2 Schema-on-read and *stale* metadata
For file-based tables Spark caches the **list of files** in the session. When the source drops a new file, the catalog does not know about it until you `REFRESH TABLE`. This is another place where *metadata*, not data, determines what a query sees.

In [ ]:
late_file = bronze_landing / "online_retail_2011_part_0002_late_arrival.csv"
with open(file_name) as src, open(late_file, "w") as dst:      # a tiny late batch: header + 5 lines
    for i, line in enumerate(src):
        if i > 5:
            break
        dst.write(line)

print("Count with the cached file list  :", spark.table("bronze.online_retail_raw").count(), " (probably unchanged)")
spark.sql("REFRESH TABLE bronze.online_retail_raw")
print("Count after REFRESH TABLE        :", spark.table("bronze.online_retail_raw").count())

# remove the demo file so it does not duplicate rows downstream
late_file.unlink()
spark.sql("REFRESH TABLE bronze.online_retail_raw")
print("Count after removing the file    :", spark.table("bronze.online_retail_raw").count())

## Part 5 — Silver: clean, typed, conformed, documented

Silver applies **technical** rules (no business opinions yet):
* consistent `snake_case` names,
* real data types: timestamp, date, `INT` customer id, `DECIMAL` for money (never float for money!),
* trimming and default values, removal of exact duplicate lines,
* **flags** that describe the data (`is_cancellation`, `is_product`) — Silver labels rows, Gold decides what to do with them,
* **audit columns** (`_source_file`, `_ingested_at`) — operational metadata stored *in* the data, which enables lineage back to the exact raw file.

In [ ]:
bronze_df = spark.table("bronze.online_retail_raw")

business_columns = ["invoice_no", "stock_code", "product_description", "quantity",
                    "invoice_ts", "unit_price", "customer_id", "country"]

silver_df = (bronze_df
    .select(
        F.trim("InvoiceNo").alias("invoice_no"),
        F.upper(F.trim("StockCode")).alias("stock_code"),
        F.coalesce(F.trim("Description"), F.lit("UNKNOWN")).alias("product_description"),
        F.col("Quantity").alias("quantity"),
        F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm").alias("invoice_ts"),
        F.col("UnitPrice").cast("decimal(10,3)").alias("unit_price"),
        F.col("CustomerID").cast("int").alias("customer_id"),
        F.trim("Country").alias("country"),
        F.input_file_name().alias("_source_file"))
    .dropDuplicates(business_columns)
    .withColumn("invoice_date",    F.to_date("invoice_ts"))
    .withColumn("invoice_month",   F.date_format("invoice_ts", "yyyy-MM"))
    .withColumn("line_amount",     F.round(F.col("quantity") * F.col("unit_price"), 2).cast("decimal(12,2)"))
    .withColumn("is_cancellation", F.col("invoice_no").startswith("C"))
    .withColumn("is_product",      F.col("stock_code").rlike("^[0-9]"))
    .withColumn("_ingested_at",    F.current_timestamp())
)

print("bronze rows:", bronze_df.count(), "| silver rows after de-duplication:", silver_df.count())
silver_df.printSchema()

In [ ]:
# A MANAGED table: the platform produces this data, so the platform owns it.
(silver_df.write
          .mode("overwrite")
          .format("parquet")
          .partitionBy("invoice_month")
          .saveAsTable("silver.sales_lines"))

tree(table_location("silver.sales_lines"), max_files=1)

### 5.1 Attach business metadata
Comments and properties are changed with `ALTER` / `COMMENT ON` statements. These are **metadata-only** operations: the Parquet files are not touched.

In [ ]:
column_comments = {
    "invoice_no":          "Invoice (order) number. A leading C marks a cancellation invoice",
    "stock_code":          "Upper-cased product code",
    "product_description": "Trimmed product description; UNKNOWN when missing in the source",
    "quantity":            "Units on the line; negative for cancellations and returns",
    "invoice_ts":          "Invoice timestamp parsed from the source text (UK local time)",
    "unit_price":          "Unit price in GBP, DECIMAL(10,3)",
    "customer_id":         "Customer identifier as INT; NULL for guest checkouts",
    "country":             "Ship-to country",
    "_source_file":        "AUDIT: raw file this row was read from (lineage to bronze)",
    "invoice_date":        "Calendar date of the invoice",
    "line_amount":         "quantity x unit_price in GBP, rounded to 2 decimals",
    "is_cancellation":     "FLAG: true when invoice_no starts with C",
    "is_product":          "FLAG: true when stock_code starts with a digit (real product, not postage/fees/adjustments)",
    "_ingested_at":        "AUDIT: timestamp when this row was written to silver",
}

for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE silver.sales_lines ALTER COLUMN {col} COMMENT '{comment}'")

spark.sql("COMMENT ON TABLE silver.sales_lines IS "
          "'Cleaned online-retail invoice lines. One row per distinct invoice line. Flags describe rows, nothing is filtered out'")

spark.sql("""
ALTER TABLE silver.sales_lines SET TBLPROPERTIES (
    'layer'             = 'silver',
    'grain'             = 'one row per distinct invoice line',
    'upstream'          = 'bronze.online_retail_raw',
    'transformations'   = 'rename to snake_case; cast types; trim; default description; drop exact duplicates; add flags and audit columns',
    'data_owner'        = 'data-engineering-class',
    'contains_pii'      = 'false',
    'refresh_frequency' = 'daily'
)
""")

sql("DESCRIBE TABLE silver.sales_lines")

### 5.2 Operational metadata: record data-quality results *on the table*
We compute a few data-quality counts and store them as table properties, so anyone browsing the catalog sees the health of the table without running a query.

In [ ]:
dq = spark.sql("""
SELECT COUNT(*)                                             AS row_count,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
       SUM(CASE WHEN invoice_ts  IS NULL THEN 1 ELSE 0 END) AS unparsed_timestamp,
       SUM(CASE WHEN is_cancellation     THEN 1 ELSE 0 END) AS cancellation_lines,
       SUM(CASE WHEN NOT is_product      THEN 1 ELSE 0 END) AS non_product_lines,
       SUM(CASE WHEN unit_price <= 0     THEN 1 ELSE 0 END) AS non_positive_price_lines
FROM silver.sales_lines
""").first().asDict()

dq_props = ", ".join(f"'dq.{k}' = '{v}'" for k, v in dq.items())
spark.sql(f"""ALTER TABLE silver.sales_lines SET TBLPROPERTIES (
    {dq_props}, 'dq.checked_at' = '{datetime.now().isoformat(timespec="seconds")}')""")

sql("SHOW TBLPROPERTIES silver.sales_lines")

In [ ]:
# The partition list is technical metadata held in the metastore
sql("SHOW PARTITIONS silver.sales_lines")

## Part 6 — Metadata deep dive: statistics feed the optimizer

`ANALYZE TABLE` scans the data once and stores **statistics** in the metastore: table size and row count, and per-column min / max / null count / distinct count. The cost-based optimizer uses them (e.g. to decide whether a table is small enough to broadcast in a join), and catalog tools show them to users.

In [ ]:
print("Before ANALYZE:")
display(table_summary("silver.sales_lines", keys=("Statistics",)))

spark.sql("ANALYZE TABLE silver.sales_lines COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE silver.sales_lines COMPUTE STATISTICS "
          "FOR COLUMNS quantity, unit_price, customer_id, country, invoice_date")

print("After ANALYZE:")
table_summary("silver.sales_lines", keys=("Statistics",))

In [ ]:
# Column-level statistics, read straight from the metastore (no data scan)
sql("DESCRIBE EXTENDED silver.sales_lines quantity")

In [ ]:
# The optimizer's view: estimated size and row count come from the metastore statistics
spark.conf.set("spark.sql.cbo.enabled", "true")      # use column stats + row counts in planning
spark.table("silver.sales_lines").filter("country = 'France'").explain(mode="cost")

In [ ]:
# All the metadata above, regenerated as DDL — the table's "birth certificate"
print(spark.sql("SHOW CREATE TABLE silver.sales_lines").first()[0])

## Part 7 — Gold: the semantic layer

Silver is correct, but it still speaks the language of the *source system*: `stock_code`, `is_product`, `line_amount`, cancellations mixed with sales. Ask three analysts "what was revenue last month?" against Silver and you can easily get three different answers.

The **semantic layer** fixes that by publishing:
* **Dimensions** — the "by what" of analysis: `dim_date`, `dim_product`, `dim_customer`.
* **Facts** — the "how much", at a declared **grain**, with business rules already applied: `fact_sales`.
* **Metrics** — named, governed calculations: `gross_revenue`, `net_revenue`, `average_order_value`, … in `sales_metrics_monthly`, documented in a `metric_definitions` registry.

We build the semantic layer from **views**. A view stores **only a SQL definition in the metastore** — no data — so the business logic lives in one place, is always up to date with Silver, and every column carries a business description.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_date (
    date_key          COMMENT 'Calendar date. Join key to fact_sales.invoice_date',
    calendar_year     COMMENT 'Calendar year',
    calendar_quarter  COMMENT 'Calendar quarter 1-4',
    month_number      COMMENT 'Month number 1-12',
    month_name        COMMENT 'Month name, e.g. September',
    year_month        COMMENT 'Reporting month in yyyy-MM format',
    day_name          COMMENT 'Day of week, e.g. Monday',
    is_weekend        COMMENT 'True on Saturday and Sunday'
)
COMMENT 'Conformed calendar dimension: every day between the first and the last invoice'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per calendar day', 'upstream' = 'silver.sales_lines')
AS
SELECT d                        AS date_key,
       year(d)                  AS calendar_year,
       quarter(d)               AS calendar_quarter,
       month(d)                 AS month_number,
       date_format(d, 'MMMM')   AS month_name,
       date_format(d, 'yyyy-MM') AS year_month,
       date_format(d, 'EEEE')   AS day_name,
       dayofweek(d) IN (1, 7)   AS is_weekend
FROM (
    SELECT explode(sequence(min_d, max_d, INTERVAL 1 DAY)) AS d
    FROM (SELECT MIN(invoice_date) AS min_d, MAX(invoice_date) AS max_d FROM silver.sales_lines) AS bounds
) AS days
""")
sql("SELECT * FROM gold.dim_date ORDER BY date_key", 5)

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_product (
    product_code     COMMENT 'Code that identifies a sellable product',
    product_name     COMMENT 'Most recent description used for the product',
    list_price       COMMENT 'Most recent unit price charged, GBP',
    first_sold_date  COMMENT 'Date of the first invoice line for the product',
    last_sold_date   COMMENT 'Date of the most recent invoice line for the product'
)
COMMENT 'Product dimension: one row per sellable product. Postage, fees and manual adjustments are excluded'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per product_code', 'upstream' = 'silver.sales_lines',
               'business_rules' = 'is_product = true')
AS
SELECT stock_code                               AS product_code,
       max_by(product_description, invoice_ts)  AS product_name,
       max_by(unit_price, invoice_ts)           AS list_price,
       MIN(invoice_date)                        AS first_sold_date,
       MAX(invoice_date)                        AS last_sold_date
FROM silver.sales_lines
WHERE is_product
GROUP BY stock_code
""")
sql("SELECT * FROM gold.dim_product ORDER BY product_code", 5)

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_customer (
    customer_key         COMMENT 'Customer identifier. Guest checkouts are not in this dimension',
    home_country         COMMENT 'Country of the most recent purchase',
    first_purchase_date  COMMENT 'Date of first purchase',
    last_purchase_date   COMMENT 'Date of most recent purchase',
    lifetime_orders      COMMENT 'Number of distinct non-cancelled invoices',
    customer_type        COMMENT 'Repeat when lifetime_orders is 2 or more, otherwise One-time'
)
COMMENT 'Customer dimension: one row per identified customer'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per customer_key', 'upstream' = 'silver.sales_lines',
               'contains_pii' = 'false')
AS
SELECT customer_id                                                        AS customer_key,
       max_by(country, invoice_ts)                                        AS home_country,
       MIN(invoice_date)                                                  AS first_purchase_date,
       MAX(invoice_date)                                                  AS last_purchase_date,
       COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END)  AS lifetime_orders,
       CASE WHEN COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END) >= 2
            THEN 'Repeat' ELSE 'One-time' END                             AS customer_type
FROM silver.sales_lines
WHERE customer_id IS NOT NULL AND is_product
GROUP BY customer_id
""")
sql("SELECT customer_type, COUNT(*) AS customers FROM gold.dim_customer GROUP BY customer_type")

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.fact_sales (
    invoice_number     COMMENT 'Invoice (order) identifier',
    invoice_date       COMMENT 'Invoice date. Joins to dim_date.date_key',
    invoice_timestamp  COMMENT 'Invoice date and time',
    product_code       COMMENT 'Joins to dim_product.product_code',
    customer_key       COMMENT 'Joins to dim_customer.customer_key. NULL for guest checkouts',
    country            COMMENT 'Ship-to country',
    quantity           COMMENT 'Units sold; negative on returns',
    unit_price         COMMENT 'Unit price in GBP',
    sales_amount       COMMENT 'quantity x unit_price in GBP; negative on returns',
    is_return          COMMENT 'True when the line reverses an earlier sale (cancellation or negative quantity)',
    is_guest_checkout  COMMENT 'True when no customer id was captured'
)
COMMENT 'Sales fact at invoice-line grain with business rules applied: sellable products only, positive prices only'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'fact',
               'grain' = 'one row per invoice line', 'upstream' = 'silver.sales_lines',
               'business_rules' = 'is_product = true AND unit_price > 0')
AS
SELECT invoice_no                         AS invoice_number,
       invoice_date,
       invoice_ts                         AS invoice_timestamp,
       stock_code                         AS product_code,
       customer_id                        AS customer_key,
       country,
       quantity,
       unit_price,
       line_amount                        AS sales_amount,
       is_cancellation OR quantity < 0    AS is_return,
       customer_id IS NULL                AS is_guest_checkout
FROM silver.sales_lines
WHERE is_product AND unit_price > 0
""")
sql("SELECT * FROM gold.fact_sales", 5)

### 7.1 Governed metrics
This view is the **single source of truth** for sales KPIs. Pay attention to the *additivity* of each metric — it is part of its metadata:
* **Additive** (`gross_revenue`, `net_revenue`, `units_sold`, `orders`): safe to `SUM` across months and countries.
* **Non-additive** (`average_order_value`, `return_rate`): ratios — never sum or average them; recompute from their additive parts.
* **Distinct counts** (`active_customers`): additive across countries only if a customer has one country, and **not** additive across months (the same customer can buy in several months).

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.sales_metrics_monthly (
    year_month           COMMENT 'Reporting month, yyyy-MM',
    country              COMMENT 'Ship-to country',
    orders               COMMENT 'METRIC orders: distinct non-return invoices. Additive',
    active_customers     COMMENT 'METRIC active_customers: distinct identified customers with a purchase. NOT additive across months',
    units_sold           COMMENT 'METRIC units_sold: units on non-return lines. Additive',
    gross_revenue        COMMENT 'METRIC gross_revenue: sales_amount of non-return lines, GBP. Additive',
    returns_amount       COMMENT 'METRIC returns_amount: value of returned goods as a positive number, GBP. Additive',
    net_revenue          COMMENT 'METRIC net_revenue: gross_revenue minus returns_amount, GBP. Additive',
    average_order_value  COMMENT 'METRIC average_order_value: gross_revenue / orders. NON-ADDITIVE',
    return_rate          COMMENT 'METRIC return_rate: returns_amount / gross_revenue. NON-ADDITIVE'
)
COMMENT 'Governed sales KPIs at month x country grain. The single source of truth for revenue reporting'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'metrics',
               'grain' = 'one row per year_month and country',
               'upstream' = 'gold.fact_sales, gold.dim_date',
               'metric_registry' = 'gold.metric_definitions')
AS
SELECT d.year_month,
       f.country,
       COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.invoice_number END)  AS orders,
       COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.customer_key END)    AS active_customers,
       SUM(CASE WHEN NOT f.is_return THEN f.quantity ELSE 0 END)            AS units_sold,
       SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END)        AS gross_revenue,
       -SUM(CASE WHEN f.is_return THEN f.sales_amount ELSE 0 END)           AS returns_amount,
       SUM(f.sales_amount)                                                  AS net_revenue,
       ROUND(SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END)
             / NULLIF(COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.invoice_number END), 0), 2)
                                                                            AS average_order_value,
       ROUND(-SUM(CASE WHEN f.is_return THEN f.sales_amount ELSE 0 END)
             / NULLIF(SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END), 0), 4)
                                                                            AS return_rate
FROM gold.fact_sales AS f
JOIN gold.dim_date   AS d ON f.invoice_date = d.date_key
GROUP BY d.year_month, f.country
""")
sql("SELECT * FROM gold.sales_metrics_monthly ORDER BY year_month, net_revenue DESC", 10)

### 7.2 A metric registry — business metadata as a table
Column comments are good, but a semantic layer usually also keeps a **registry of metric definitions** that BI tools, documentation sites and people can read. This one is a small managed table.

In [ ]:
metric_rows = [
    ("orders", "Orders", "Number of distinct invoices that are not returns",
     "COUNT(DISTINCT invoice_number) FILTER (WHERE NOT is_return)", "additive", "count"),
    ("active_customers", "Active customers", "Distinct identified customers with at least one purchase in the period",
     "COUNT(DISTINCT customer_key) FILTER (WHERE NOT is_return)", "distinct count - not additive across periods", "count"),
    ("units_sold", "Units sold", "Units on non-return lines",
     "SUM(quantity) FILTER (WHERE NOT is_return)", "additive", "units"),
    ("gross_revenue", "Gross revenue", "Value of goods sold before returns. Excludes postage, fees and adjustments",
     "SUM(sales_amount) FILTER (WHERE NOT is_return)", "additive", "GBP"),
    ("returns_amount", "Returns", "Value of goods returned or cancelled, as a positive number",
     "-SUM(sales_amount) FILTER (WHERE is_return)", "additive", "GBP"),
    ("net_revenue", "Net revenue", "Gross revenue minus returns",
     "SUM(sales_amount)", "additive", "GBP"),
    ("average_order_value", "Average order value", "Average gross value of an order",
     "gross_revenue / orders", "non-additive (ratio)", "GBP"),
    ("return_rate", "Return rate", "Share of gross revenue that was returned",
     "returns_amount / gross_revenue", "non-additive (ratio)", "ratio"),
]
metric_schema = "metric_name STRING, display_name STRING, business_definition STRING, " \
                "formula_sql STRING, additivity STRING, unit STRING"

(spark.createDataFrame(metric_rows, metric_schema)
      .withColumn("source_object", F.lit("gold.fact_sales"))
      .withColumn("metric_owner",  F.lit("finance-analytics"))
      .withColumn("certified",     F.lit(True))
      .write.mode("overwrite").format("parquet")
      .saveAsTable("gold.metric_definitions"))

spark.sql("COMMENT ON TABLE gold.metric_definitions IS 'Registry of governed metric definitions used by the semantic layer'")
sql("SELECT * FROM gold.metric_definitions")

### 7.3 Using the semantic layer
Business questions become short, readable queries in business vocabulary — no need to know about cancellation prefixes or postage stock codes.

In [ ]:
# Q1: Which countries generate the most net revenue (full period)?
sql("""
SELECT country,
       SUM(net_revenue)                                AS net_revenue,
       SUM(orders)                                     AS orders,
       ROUND(SUM(gross_revenue) / SUM(orders), 2)      AS average_order_value
FROM gold.sales_metrics_monthly
GROUP BY country
ORDER BY net_revenue DESC
""", 10)

In [ ]:
# Q2: Monthly trend of gross revenue, returns and net revenue
sql("""
SELECT year_month,
       SUM(gross_revenue)  AS gross_revenue,
       SUM(returns_amount) AS returns_amount,
       SUM(net_revenue)    AS net_revenue
FROM gold.sales_metrics_monthly
GROUP BY year_month
ORDER BY year_month
""")

### 7.4 Why bother? "Naive" vs. governed numbers
An analyst who goes straight to the raw table and writes the obvious query gets a different "revenue" and a different "number of orders".

In [ ]:
sql("""
SELECT 'Naive: SUM(Quantity*UnitPrice) on bronze' AS approach,
       ROUND(SUM(Quantity * UnitPrice), 2)       AS revenue,
       COUNT(DISTINCT InvoiceNo)                 AS orders
FROM bronze.online_retail_raw
UNION ALL
SELECT 'Governed: gross_revenue (gold)', CAST(SUM(gross_revenue) AS DOUBLE), SUM(orders)
FROM gold.sales_metrics_monthly
UNION ALL
SELECT 'Governed: net_revenue (gold)',   CAST(SUM(net_revenue) AS DOUBLE),   SUM(orders)
FROM gold.sales_metrics_monthly
""")

In [ ]:
# Where does the difference come from? The business rules, made explicit
sql("""
SELECT CASE WHEN NOT is_product                    THEN '1. non-product lines (postage, fees, adjustments)'
            WHEN unit_price <= 0                   THEN '2. zero-price lines'
            WHEN is_cancellation OR quantity < 0   THEN '3. returns / cancellations'
            ELSE                                        '4. regular product sales'
       END                AS bucket,
       COUNT(*)           AS lines,
       SUM(line_amount)   AS amount_gbp
FROM silver.sales_lines
GROUP BY 1
ORDER BY 1
""")

### 7.5 The non-additive trap
`average_order_value` is stored per month × country. Averaging those averages weights a country with 1 order the same as the UK with hundreds. The metadata ("NON-ADDITIVE") tells the analyst what to do instead.

In [ ]:
sql("""
SELECT 'WRONG: AVG(average_order_value)'          AS method, ROUND(AVG(average_order_value), 2)          AS aov
FROM gold.sales_metrics_monthly
UNION ALL
SELECT 'RIGHT: SUM(gross_revenue) / SUM(orders)', ROUND(SUM(gross_revenue) / SUM(orders), 2)
FROM gold.sales_metrics_monthly
""")

### 7.6 Views are pure metadata
A view has `Type: VIEW`, no `Location`, and its SQL text stored in the metastore. Look at `gold.db` in the warehouse: only the **managed table** `metric_definitions` has files — the five views have none.

In [ ]:
table_summary("gold.sales_metrics_monthly", keys=("Type", "Comment", "Table Properties", "View Text"))

In [ ]:
tree(WAREHOUSE_DIR / "gold.db", max_files=1)
catalog_tables("gold")

In [ ]:
# A permanent view is stored in the metastore, so it may NOT depend on a session-only temp view
try:
    spark.sql("CREATE OR REPLACE VIEW gold.bad_view AS SELECT * FROM sales")   # 'sales' is our temp view
except Exception as e:
    print("Refused ->", str(e).splitlines()[0][:200])

### 7.7 Virtual vs. materialized semantic layer
Views are always fresh but recompute on every query. For dashboards we can **materialize** a snapshot as a managed table (CTAS). Trade-off: speed vs. freshness — and now the metadata must also say *when* it was refreshed.

> Vanilla Spark Parquet tables do not support `CREATE OR REPLACE TABLE` (that needs Delta/Iceberg), so we drop and re-create.

In [ ]:
spark.sql("DROP TABLE IF EXISTS gold.agg_sales_monthly")
spark.sql(f"""
CREATE TABLE gold.agg_sales_monthly
USING PARQUET
COMMENT 'Materialized snapshot of gold.sales_metrics_monthly for fast dashboards'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'aggregate',
               'upstream' = 'gold.sales_metrics_monthly',
               'refresh_frequency' = 'daily',
               'last_refreshed_at' = '{datetime.now().isoformat(timespec="seconds")}')
AS SELECT * FROM gold.sales_metrics_monthly
""")

pd.concat({t: table_summary(f"gold.{t}", keys=("Type", "Location"))["value"]
           for t in ["sales_metrics_monthly", "agg_sales_monthly"]}, axis=1)

## Part 8 — Metadata as data: generate a data dictionary

Everything we attached is queryable through the catalog API. That means documentation can be **generated** instead of hand-written — and kept in its own `governance` database. This is, in miniature, what tools like DataHub, Amundsen, OpenMetadata, Unity Catalog or Glue do.

In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS governance
COMMENT 'Metadata about our data: data dictionary and catalog snapshots'
WITH DBPROPERTIES ('layer' = 'governance', 'data_owner' = 'data-engineering-class')
""")

def build_data_dictionary(databases):
    rows = []
    for db in databases:
        for t in spark.catalog.listTables(db):
            if t.isTemporary:
                continue
            fqtn  = f"{db}.{t.name}"
            props = {r.key: r.value for r in spark.sql(f"SHOW TBLPROPERTIES {fqtn}").collect()}
            for c in spark.catalog.listColumns(fqtn):
                rows.append((db, t.name, t.tableType, t.description,
                             props.get("layer"), props.get("grain"), props.get("upstream"),
                             c.name, c.dataType, c.nullable, c.isPartition, c.description))
    schema = ("database STRING, table_name STRING, table_type STRING, table_description STRING, "
              "layer STRING, grain STRING, upstream STRING, "
              "column_name STRING, data_type STRING, is_nullable BOOLEAN, is_partition BOOLEAN, "
              "column_description STRING")
    return spark.createDataFrame(rows, schema).withColumn("snapshot_at", F.current_timestamp())

data_dictionary = build_data_dictionary(["bronze", "silver", "gold"])
data_dictionary.write.mode("overwrite").format("parquet").saveAsTable("governance.data_dictionary")

sql("SELECT database, table_name, table_type, column_name, data_type, column_description "
    "FROM governance.data_dictionary ORDER BY database, table_name", 200)

In [ ]:
# Metadata quality: how well documented is each layer?
# Expect gaps on purpose: gold.metric_definitions columns and silver's partition column
# (invoice_month) were never commented. See Exercise 5.
sql("""
SELECT database,
       table_name,
       table_type,
       COUNT(*)                                                          AS columns,
       SUM(CASE WHEN column_description IS NULL OR column_description = '' THEN 1 ELSE 0 END) AS undocumented_columns
FROM governance.data_dictionary
GROUP BY database, table_name, table_type
ORDER BY database, table_name
""")

In [ ]:
# Lineage, straight from the 'upstream' properties we recorded
sql("""
SELECT DISTINCT upstream AS from_object, CONCAT(database, '.', table_name) AS to_object, table_type
FROM governance.data_dictionary
WHERE upstream IS NOT NULL
ORDER BY to_object
""")

In [ ]:
# Hand it to non-Spark users (wiki, spreadsheet, catalog tool)
data_dictionary.toPandas().to_csv("data_dictionary.csv", index=False)
print("Wrote", PROJECT_DIR / "data_dictionary.csv")

## Part 9 — Wrap-up

### 9.1 The whole catalog at a glance

In [ ]:
pd.concat([catalog_tables(db) for db in DEMO_DATABASES], ignore_index=True)[
    ["namespace", "name", "tableType", "description"]]

In [ ]:
print("Data (managed tables) in the warehouse:")
tree(WAREHOUSE_DIR, max_files=0)
print("\nData we own (external tables):")
tree(EXTERNAL_DIR, max_files=0)
print("\nRaw landing zone (bronze external table):")
tree(LANDING_DIR)

In [ ]:
spark.stop()

### 9.2 Proof of persistence
1. **Restart the kernel** (Kernel → Restart). This throws away the JVM, every DataFrame and every temp view.
2. Run the **setup cell (0.1)** and the **helpers cell (0.2)** only. **Do not run 0.3 (reset).**
3. Run the cell below.

Everything that was in the metastore comes back: databases, tables, comments, properties, statistics, view definitions. The temp views `sales` and `global_temp.sales_global` are gone, because they never were in the metastore.

In [ ]:
# Run ONLY after a kernel restart + cells 0.1 and 0.2 (in this session Spark was just stopped)
for db in ["lab", "bronze", "silver", "gold", "governance"]:
    print(f"{db:11s}", [(t.name, t.tableType) for t in spark.catalog.listTables(db) if not t.isTemporary])

print("\nTemp view 'sales' still there?", spark.catalog.tableExists("sales"))
sql("SELECT country, SUM(net_revenue) AS net_revenue FROM gold.sales_metrics_monthly "
    "GROUP BY country ORDER BY net_revenue DESC", 5)

## Part 10 — Exercises

1. **In-memory catalog.** Restart the kernel and create the session *without* `.enableHiveSupport()`. Run `spark.catalog.listDatabases()`. Where did `silver` go? Do the Parquet files under `spark-warehouse/silver.db/` still exist? What does this tell you about the relationship between files and metadata?
2. **Experiment 4 for real.** Create an external table `lab.shadow` whose `LOCATION` is the folder of `lab.sales_managed_v2`. Drop `lab.sales_managed_v2`, then query `lab.shadow`. Explain the result.
3. **New metric.** Add `distinct_products_sold` to `gold.sales_metrics_monthly` *and* to `gold.metric_definitions`. Is it additive across months? Across countries?
4. **Schema evolution.** The source starts sending a 9th column `Channel`. What changes in Bronze (schema-on-read), Silver and Gold? Which layers break, which do not?
5. **Metadata coverage.** Use `governance.data_dictionary` to list every Gold column without a description. Fix them and regenerate the dictionary.
6. **Stale statistics.** Append a second copy of a month to `silver.sales_lines` with `INSERT INTO`. Are the statistics from `ANALYZE TABLE` still correct? How would you detect and fix it in a pipeline?
7. **Sensitive data.** If `CustomerID` were an email address, which metadata (properties, comments, a separate column-level tag table) would you add, and in which layer would you mask it?

### Recap
| Concept | One-line definition | Where we saw it |
|---|---|---|
| Metadata | Data about data: technical, business, operational | `DESCRIBE`, comments, `TBLPROPERTIES`, `ANALYZE` |
| Metastore | The database that persists metadata | `./metastore_db`, survived the restart in 9.2 |
| Catalog | API / namespace over the metastore | `spark.catalog.*`, `SHOW …`, data dictionary |
| Managed table | Spark owns metadata **and** files | `lab.sales_managed`, `silver.sales_lines` |
| External table | Spark owns metadata only | `lab.sales_external`, `bronze.online_retail_raw` |
| Temp view | Session-only name for a query plan | `sales`, `global_temp.sales_global` |
| Semantic layer | Business-friendly, governed model over physical tables | `gold.dim_*`, `gold.fact_sales`, `gold.sales_metrics_monthly`, `gold.metric_definitions` |